# Gold - ecommerce_pedidos

Notebook para criação de tabelas Gold em Delta Lake e réplica opcional para SQL Server/Azure, mantendo padrão de consumo analítico via Looker.

Este notebook assume que as tabelas Silver e `squad1.dq_monitoring_logs` já foram criadas em Delta.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SCHEMA = "squad1"
DQ_LOGS_TABLE = f"{SCHEMA}.dq_monitoring_logs"

# Ative como True somente se desejar replicar as Golds para SQL Server/Azure.
REPLICAR_SQLSERVER = False

# Caso use SQL Server, o notebook config precisa ter carregado:
# JDBC_HOSTNAME, JDBC_DATABASE, JDBC_USERNAME, JDBC_PASSWORD


In [0]:
def tabela_delta_existe(nome_tabela: str) -> bool:
    return spark.catalog.tableExists(nome_tabela)


def salvar_gold_delta(df, tabela_destino: str, chaves_merge=None):
    """
    Salva a Gold como tabela Delta gerenciada.
    Para Gold agregada, usamos overwrite para recalcular o snapshot analítico.
    Isso evita duplicidade mesmo com múltiplas execuções.
    """
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )
    print(f"Tabela Delta atualizada: {tabela_destino}")


def replicar_sqlserver(df, tabela_destino: str):
    """
    Replica para SQL Server usando overwrite, pois Gold agregada deve representar
    o estado analítico atual, não append incremental bruto.
    """
    if not REPLICAR_SQLSERVER:
        print(f"Réplica SQL Server desativada para {tabela_destino}")
        return

    (
        df.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )
    print(f"Tabela replicada para SQL Server: {tabela_destino}")


def criar_gold_dq_resumo_por_regra(nome_tabela_silver: str):
    df_logs = spark.table(DQ_LOGS_TABLE)

    return (
        df_logs
        .filter(F.col("tabela") == nome_tabela_silver)
        .withColumn("data_referencia", F.to_date("timestamp_execucao"))
        .groupBy("data_referencia", "tabela", "regra", "severidade")
        .agg(
            F.sum("qtd_registros_falhos").alias("qtd_registros_falhos"),
            F.sum("qtd_registros_total").alias("qtd_registros_total")
        )
        .withColumn(
            "perc_falha",
            F.when(F.col("qtd_registros_total") > 0,
                   F.round((F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .withColumn("gold_processed_at", F.current_timestamp())
    )


def criar_gold_dq_resumo_por_tabela(df_silver, nome_tabela_silver: str, colunas_falha: list):
    cond_falha = None
    for c in colunas_falha:
        expr = F.coalesce(F.col(c), F.lit(False)) == True
        cond_falha = expr if cond_falha is None else (cond_falha | expr)

    return (
        df_silver
        .withColumn("hora_referencia", F.date_trunc("hour", F.col("silver_processed_at")))
        .withColumn("linha_com_falha", cond_falha)
        .groupBy("hora_referencia")
        .agg(
            F.count("*").alias("qtd_registros_total"),
            F.sum(F.when(F.col("linha_com_falha"), 1).otherwise(0)).alias("qtd_registros_com_falha")
        )
        .withColumn("tabela", F.lit(nome_tabela_silver))
        .withColumn("qtd_registros_limpos", F.col("qtd_registros_total") - F.col("qtd_registros_com_falha"))
        .withColumn(
            "perc_registros_limpos",
            F.when(F.col("qtd_registros_total") > 0,
                   F.round((F.col("qtd_registros_limpos") / F.col("qtd_registros_total")) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .withColumn("gold_processed_at", F.current_timestamp())
        .select(
            "hora_referencia", "tabela", "qtd_registros_total",
            "qtd_registros_com_falha", "qtd_registros_limpos",
            "perc_registros_limpos", "gold_processed_at"
        )
    )


## Leitura das tabelas Silver e logs

In [0]:
SILVER_PEDIDOS = "squad1.silver_ecommerce_pedidos"

if not tabela_delta_existe(SILVER_PEDIDOS):
    raise Exception(f"Tabela não encontrada: {SILVER_PEDIDOS}")
if not tabela_delta_existe(DQ_LOGS_TABLE):
    raise Exception(f"Tabela não encontrada: {DQ_LOGS_TABLE}")

df_pedidos = spark.table(SILVER_PEDIDOS)
display(df_pedidos.limit(5))

## Gold DQ - resumo por regra e por tabela

In [0]:
colunas_falha_pedidos = [
    "r1_id_pedido_falhou",
    "r2_id_cliente_falhou",
    "r3_status_pedido_falhou",
    "r4_valor_total_falhou",
    "r5_metodo_pagamento_falhou",
    "r6_dt_status_anterior_pedido_falhou",
    "r7_frete_gratis_falhou",
    "r8_cancelado_com_rastreamento_falhou",
    "r9_entrega_muito_rapida_falhou",
    "r10_endereco_entrega_fk_falhou"
]
colunas_falha_pedidos = [c for c in colunas_falha_pedidos if c in df_pedidos.columns]
if len(colunas_falha_pedidos) == 0:
    raise Exception("Nenhuma coluna de falha encontrada na Silver de pedidos.")

df_gold_dq_regra_pedidos = criar_gold_dq_resumo_por_regra("silver_ecommerce_pedidos")
df_gold_dq_tabela_pedidos = criar_gold_dq_resumo_por_tabela(df_pedidos, "silver_ecommerce_pedidos", colunas_falha_pedidos)

salvar_gold_delta(df_gold_dq_regra_pedidos, "squad1.gold_pedidos_dq_resumo_por_regra")
salvar_gold_delta(df_gold_dq_tabela_pedidos, "squad1.gold_pedidos_dq_resumo_por_tabela")
replicar_sqlserver(df_gold_dq_regra_pedidos, "squad1.gold_pedidos_dq_resumo_por_regra")
replicar_sqlserver(df_gold_dq_tabela_pedidos, "squad1.gold_pedidos_dq_resumo_por_tabela")

## KPIs específicos de pedidos

In [0]:
df_pedidos_base = (
    df_pedidos
    .withColumn("dt_pedido_ts", F.to_timestamp("dt_pedido"))
    .withColumn("data_referencia", F.to_date("dt_pedido"))
)

# Fallback caso dt_pedido não exista ou venha nulo
if "dt_pedido" not in df_pedidos.columns:
    df_pedidos_base = df_pedidos.withColumn("data_referencia", F.to_date("silver_processed_at"))

df_gold_pedidos_kpis = (
    df_pedidos_base
    .groupBy("data_referencia")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.round(F.sum("valor_total"), 2).alias("valor_total_vendas"),
        F.round(F.avg("valor_total"), 2).alias("ticket_medio"),
        F.round(F.sum("valor_frete"), 2).alias("valor_frete_total"),
        F.round(F.avg("valor_frete"), 2).alias("frete_medio"),
        F.sum(F.when(F.col("status_pedido") == "Cancelado", 1).otherwise(0)).alias("qtd_cancelados"),
        F.sum(F.when(F.col("status_pedido") == "Entregue", 1).otherwise(0)).alias("qtd_entregues"),
        F.sum(F.when(F.col("status_pedido") == "Enviado", 1).otherwise(0)).alias("qtd_enviados"),
        F.sum(F.when(F.col("status_pedido") == "Processando", 1).otherwise(0)).alias("qtd_processando")
    )
    .withColumn("taxa_cancelamento", F.round((F.col("qtd_cancelados") / F.col("qtd_pedidos")) * 100, 2))
    .withColumn("taxa_entrega", F.round((F.col("qtd_entregues") / F.col("qtd_pedidos")) * 100, 2))
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_pedidos_kpis, "squad1.gold_pedidos_kpis")
replicar_sqlserver(df_gold_pedidos_kpis, "squad1.gold_pedidos_kpis")
display(df_gold_pedidos_kpis)

## Pedidos por status e método de pagamento

In [0]:
df_gold_pedidos_status = (
    df_pedidos_base
    .groupBy("data_referencia", "status_pedido")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.round(F.sum("valor_total"), 2).alias("valor_total")
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_pedidos_status, "squad1.gold_pedidos_por_status")
replicar_sqlserver(df_gold_pedidos_status, "squad1.gold_pedidos_por_status")

df_gold_pedidos_pagamento = (
    df_pedidos_base
    .groupBy("data_referencia", "metodo_pagamento")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.round(F.sum("valor_total"), 2).alias("valor_total")
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_pedidos_pagamento, "squad1.gold_pedidos_por_pagamento")
replicar_sqlserver(df_gold_pedidos_pagamento, "squad1.gold_pedidos_por_pagamento")

In [0]:
tabelas_gold_criadas = ['squad1.gold_pedidos_dq_resumo_por_regra', 'squad1.gold_pedidos_dq_resumo_por_tabela', 'squad1.gold_pedidos_kpis', 'squad1.gold_pedidos_por_status', 'squad1.gold_pedidos_por_pagamento']

In [0]:
# Validação final das tabelas criadas
for tabela in tabelas_gold_criadas:
    print(f"\n{tabela}")
    spark.sql(f"DESCRIBE DETAIL {tabela}").select("format", "numFiles", "sizeInBytes", "location").show(truncate=False)
    print(f"Registros: {spark.table(tabela).count()}")
    display(spark.table(tabela).limit(10))
